# 00 — Data Audit

Use this notebook before training. It checks configured paths, summarizes YOLO labels, and visualizes random labeled images. It does not modify data.

In [ ]:
from pathlib import Path
import os
import random
import sys

def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "config" / "competition.yaml").is_file():
            return path
    raise FileNotFoundError("Could not find repo root containing config/competition.yaml")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print(REPO_ROOT)

In [ ]:
import pandas as pd
import yaml
from ua_detrac_starter.config import config_path, dataset_split_path, load_config, load_yaml

cfg, cfg_path = load_config("config/competition.yaml")
dataset_yaml = config_path(cfg, "dataset_yaml", "config/dataset.yaml")
dataset_cfg = load_yaml(dataset_yaml)

train_images = dataset_split_path(dataset_yaml, dataset_cfg, "train")
val_images = dataset_split_path(dataset_yaml, dataset_cfg, "val")
test_images = dataset_split_path(dataset_yaml, dataset_cfg, "test")
sample_submission = config_path(cfg, "sample_submission", "data/competition_starter/sample_submission.csv")

paths = pd.DataFrame([
    {"name": "config", "path": cfg_path, "exists": cfg_path.is_file()},
    {"name": "dataset_yaml", "path": dataset_yaml, "exists": dataset_yaml.is_file()},
    {"name": "sample_submission", "path": sample_submission, "exists": sample_submission.is_file()},
    {"name": "train_images", "path": train_images, "exists": train_images.is_dir()},
    {"name": "train_labels", "path": train_images.parent / "labels", "exists": (train_images.parent / "labels").is_dir()},
    {"name": "val_images", "path": val_images, "exists": val_images.is_dir()},
    {"name": "val_labels", "path": val_images.parent / "labels", "exists": (val_images.parent / "labels").is_dir()},
    {"name": "test_images", "path": test_images, "exists": test_images.is_dir()},
])
paths

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(directory: Path) -> list[Path]:
    if not directory.is_dir():
        return []
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_EXTS)


def image_lookup_keys(value: str | Path) -> set[str]:
    raw_name = Path(str(value).strip()).name
    if not raw_name:
        return set()

    path = Path(raw_name)
    stem = path.stem if path.suffix else raw_name
    keys = {raw_name, stem}

    if ".rf." in stem:
        keys.add(stem.split(".rf.", 1)[0])

    for key in list(keys):
        for extension in IMAGE_EXTS:
            encoded_extension = f"_{extension.lstrip('.')}"
            if key.lower().endswith(encoded_extension):
                base = key[: -len(encoded_extension)]
                keys.add(base)
                keys.add(f"{base}{extension}")

    return {key.lower() for key in keys if key}


def build_image_index(images_dir: Path) -> dict[str, Path]:
    index: dict[str, Path] = {}
    for image_path in list_images(images_dir):
        for key in image_lookup_keys(image_path.name):
            index.setdefault(key, image_path)
    return index


def find_indexed_image(image_id: str, image_index: dict[str, Path]) -> Path | None:
    for key in image_lookup_keys(image_id):
        if key in image_index:
            return image_index[key]
    return None


split_summary = pd.DataFrame([
    {"split": "train", "images": len(list_images(train_images)), "labels": len(list((train_images.parent / "labels").glob("*.txt"))) if (train_images.parent / "labels").is_dir() else 0},
    {"split": "val", "images": len(list_images(val_images)), "labels": len(list((val_images.parent / "labels").glob("*.txt"))) if (val_images.parent / "labels").is_dir() else 0},
    {"split": "test", "images": len(list_images(test_images)), "labels": "n/a"},
])
split_summary


In [ ]:
if sample_submission.is_file():
    sample = pd.read_csv(sample_submission, dtype=str, keep_default_na=False, na_filter=False)
else:
    sample = pd.DataFrame(columns=["id", "image_id"])

sample_image_ids = sample["image_id"].astype(str).str.strip().tolist() if "image_id" in sample else []
test_index = build_image_index(test_images)
matched = [(image_id, find_indexed_image(image_id, test_index)) for image_id in sample_image_ids]
missing = [image_id for image_id, image_path in matched if image_path is None]

sample_test_summary = pd.DataFrame([
    {"check": "sample rows", "count": len(sample)},
    {"check": "test files", "count": len(list_images(test_images))},
    {"check": "sample image_ids matched to test files", "count": len(matched) - len(missing)},
    {"check": "sample image_ids missing from test files", "count": len(missing)},
])
display(sample_test_summary)

if missing:
    print("Missing examples:", missing[:10])
else:
    print("All sample image_ids match test files, including Roboflow-style *_jpg.rf.<hash>.jpg names.")

pd.DataFrame(
    [
        {"image_id": image_id, "matched_file": image_path.name if image_path else None}
        for image_id, image_path in matched[:10]
    ]
)


In [ ]:
class_names = dataset_cfg.get("names", {})
class_names = {int(key): value for key, value in class_names.items()}

def parse_label_dir(label_dir: Path, split: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    issues = []
    if not label_dir.is_dir():
        issues.append({"split": split, "label_file": str(label_dir), "line": 0, "issue": "missing label directory"})
        return pd.DataFrame(rows), pd.DataFrame(issues)

    for label_file in sorted(label_dir.glob("*.txt")):
        lines = [line.strip() for line in label_file.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not lines:
            issues.append({"split": split, "label_file": str(label_file), "line": 0, "issue": "empty label file"})
            continue

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                issues.append({"split": split, "label_file": str(label_file), "line": line_number, "issue": f"expected 5 tokens, found {len(parts)}"})
                continue
            try:
                class_id = int(parts[0])
                x_center, y_center, width, height = [float(value) for value in parts[1:]]
            except ValueError:
                issues.append({"split": split, "label_file": str(label_file), "line": line_number, "issue": "non-numeric value"})
                continue
            if class_id not in class_names:
                issues.append({"split": split, "label_file": str(label_file), "line": line_number, "issue": f"unknown class {class_id}"})
                continue
            if not all(0.0 <= value <= 1.0 for value in [x_center, y_center, width, height]):
                issues.append({"split": split, "label_file": str(label_file), "line": line_number, "issue": "coordinate outside [0, 1]"})
                continue
            rows.append({
                "split": split,
                "image_stem": label_file.stem,
                "label_file": str(label_file),
                "class_id": class_id,
                "class_name": class_names[class_id],
                "x_center": x_center,
                "y_center": y_center,
                "width": width,
                "height": height,
                "area": width * height,
            })
    return pd.DataFrame(rows), pd.DataFrame(issues)

train_labels, train_issues = parse_label_dir(train_images.parent / "labels", "train")
val_labels, val_issues = parse_label_dir(val_images.parent / "labels", "val")
labels = pd.concat([train_labels, val_labels], ignore_index=True)
issues = pd.concat([train_issues, val_issues], ignore_index=True)

print(f"valid boxes: {len(labels):,}")
print(f"issues: {len(issues):,}")
labels.head()

In [ ]:
if issues.empty:
    print("No label-format issues found.")
else:
    display(issues.head(30))

In [ ]:
if labels.empty:
    print("No labels loaded. Check the data paths above.")
else:
    display(pd.crosstab(labels["class_name"], labels["split"], margins=True))
    display(labels.groupby(["split", "class_name"])[["width", "height", "area"]].describe().round(4))

In [ ]:
import matplotlib.pyplot as plt

if not labels.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    labels["class_name"].value_counts().reindex(class_names.values()).plot(kind="bar", ax=axes[0], title="Boxes per class")
    labels["area"].clip(upper=0.2).plot(kind="hist", bins=50, ax=axes[1], title="Box area distribution (clipped at 0.2)")
    axes[0].set_xlabel("")
    axes[1].set_xlabel("normalized area")
    plt.tight_layout()

In [ ]:
from PIL import Image
from matplotlib.patches import Rectangle


def image_for_stem(images_dir: Path, stem: str) -> Path | None:
    image_index = build_image_index(images_dir)
    return find_indexed_image(stem, image_index)


def draw_yolo_boxes(image_path: Path, label_path: Path | None, ax):
    image = Image.open(image_path).convert("RGB")
    image_width, image_height = image.size
    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis("off")

    if label_path is None or not label_path.is_file():
        return
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        class_id, x_center, y_center, width, height = int(parts[0]), *[float(value) for value in parts[1:]]
        left = (x_center - width / 2) * image_width
        top = (y_center - height / 2) * image_height
        rect = Rectangle((left, top), width * image_width, height * image_height, fill=False, edgecolor="lime", linewidth=2)
        ax.add_patch(rect)
        ax.text(left, max(0, top - 3), class_names.get(class_id, class_id), color="black", backgroundcolor="lime", fontsize=8)


sample_split = "test"  # choose: train, val, or test
sample_count = 6

if sample_split == "train":
    images_dir = train_images
    labels_dir = train_images.parent / "labels"
    candidates = [(path, labels_dir / f"{path.stem}.txt") for path in list_images(images_dir)]
elif sample_split == "val":
    images_dir = val_images
    labels_dir = val_images.parent / "labels"
    candidates = [(path, labels_dir / f"{path.stem}.txt") for path in list_images(images_dir)]
elif sample_split == "test":
    images_dir = test_images
    candidates = [(path, None) for path in list_images(images_dir)]
else:
    raise ValueError("sample_split must be train, val, or test")

if sample_split in {"train", "val"}:
    candidates = [(image_path, label_path) for image_path, label_path in candidates if label_path is not None and label_path.is_file()]

if not candidates:
    print(f"No image candidates found for split={sample_split}.")
else:
    chosen = random.sample(candidates, k=min(sample_count, len(candidates)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, (image_path, label_path) in zip(axes.flat, chosen):
        draw_yolo_boxes(image_path, label_path, ax)
    for ax in axes.flat[len(chosen):]:
        ax.axis("off")
    plt.tight_layout()
